# خطة التطبيق العملي لخوارزمية SAA

هنحتاج نجهز بيئة العمل، ننفذ الخوارزمية، ونقيم أدائها.

## 📦 أولاً: تجهيز البيئة والنماذج

In [ ]:
# تثبيت المكتبات المطلوبة
# !pip install torch transformers accelerate sentencepiece
import subprocess
import sys
try:
    import lm_eval
except ImportError:
    print('Installing lm-eval (optional for evaluation)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'lm-eval', '--no-deps'])

import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
import copy
import gc

# تحميل النموذج الأساسي ونسختين محسنتين
base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# هنستخدم نسخة محسنة على الرياضيات ونسخة محسنة على الكود
math_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # استبدلها بنسخة محسنة حقيقية
code_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # استبدلها بنسخة محسنة حقيقية

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
load_dtype = torch.float16

# تحميل النماذج
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=load_dtype,
    low_cpu_mem_usage=True
)
math_model = AutoModelForCausalLM.from_pretrained(
    math_model_name,
    dtype=load_dtype,
    low_cpu_mem_usage=True
)
code_model = AutoModelForCausalLM.from_pretrained(
    code_model_name,
    dtype=load_dtype,
    low_cpu_mem_usage=True
)

# تثبيت الأوزان للقراءة فقط (لتوفير الذاكرة)
for model in [base_model, math_model, code_model]:
    for param in model.parameters():
        param.requires_grad = False

## ⚙️ ثانياً: تنفيذ خوارزمية SAA

In [ ]:
def svd_whiten_and_align(base_model, finetuned_models, top_k_ratio=0.9):
    """
    تنفيذ خوارزمية Spectral Alignment and Averaging (SAA)
    
    Args:
        base_model: النموذج الأساسي
        finetuned_models: قائمة بالنماذج المحسنة
        top_k_ratio: نسبة التباين المطلوب الاحتفاظ بها (افتراضي 90%)
    
    Returns:
        merged_model: النموذج المدمج
    """
    print("1. حساب ودمج متجهات المهام طبقة-بطبقة...")

    merged_model = copy.deepcopy(base_model)
    base_params = dict(base_model.named_parameters())
    merged_params = dict(merged_model.named_parameters())
    ft_param_maps = [dict(ft.named_parameters()) for ft in finetuned_models]

    for idx, (name, base_param) in enumerate(base_params.items()):
        print(f"معالجة: {name}, الشكل: {base_param.shape}")

        layer_vectors = []
        for ft_params in ft_param_maps:
            if name in ft_params and ft_params[name].shape == base_param.shape:
                layer_vectors.append((ft_params[name] - base_param).float())

        if len(layer_vectors) == 0:
            continue

        if base_param.dim() == 2 and base_param.shape[0] > 1 and base_param.shape[1] > 1:
            U, S, Vh = torch.linalg.svd(base_param.float(), full_matrices=False)
            V = Vh.T

            S_inv = torch.where(S > 1e-6, 1.0 / S, torch.zeros_like(S))
            whitened_vectors = []
            for lv in layer_vectors:
                Z = torch.diag(S_inv) @ U.T @ lv @ V
                whitened_vectors.append(Z.flatten())

            Z_matrix = torch.stack(whitened_vectors, dim=0)
            Z_mean = Z_matrix.mean(dim=0, keepdim=True)
            Z_centered = Z_matrix - Z_mean

            U_pca, S_pca, Vh_pca = torch.linalg.svd(Z_centered.T, full_matrices=False)

            if S_pca.numel() > 0 and (S_pca**2).sum() > 0:
                explained_var_ratio = S_pca**2 / (S_pca**2).sum()
                cumsum = torch.cumsum(explained_var_ratio, dim=0)
                k = int(torch.searchsorted(cumsum, top_k_ratio).item() + 1)
                k = min(k, len(S_pca))
            else:
                k = 0

            proj_all = Z_centered @ U_pca
            aligned_vectors = []
            for i in range(Z_matrix.shape[0]):
                proj = proj_all[i:i+1]
                aligned_proj = torch.zeros_like(proj)

                for j in range(k):
                    signs = torch.sign(proj_all[:, j])
                    majority_sign = torch.sign(signs.sum())
                    if majority_sign == 0:
                        majority_sign = 1.0
                    aligned_proj[:, j] = majority_sign * torch.abs(proj[:, j])

                aligned_z = Z_mean.squeeze(0) + (aligned_proj @ U_pca.T).squeeze(0)
                aligned_vectors.append(aligned_z)

            z_merged = torch.stack(aligned_vectors, dim=0).mean(dim=0)
            spectral_dim = S.shape[0]
            Z_merged = z_merged.reshape(spectral_dim, spectral_dim)
            merged_update = U @ torch.diag(S) @ Z_merged @ V.T
            merged_params[name].data = merged_params[name].data + merged_update.to(merged_params[name].dtype)
        else:
            stacked = torch.stack(layer_vectors, dim=0)
            merged_update = stacked.mean(dim=0)
            merged_params[name].data = merged_params[name].data + merged_update.to(merged_params[name].dtype)

        del layer_vectors
        if idx % 10 == 0:
            gc.collect()

    print("2. بناء النموذج المدمج...")
    return merged_model

# تجربة الخوارزمية
print("بدء عملية الدمج باستخدام SAA...")
merged_model_saa = svd_whiten_and_align(
    base_model,
    [math_model, code_model],
    top_k_ratio=0.9
)
print("تم الدمج بنجاح!")

## 📊 ثالثاً: تقييم أداء النموذج المدمج

للتقييم، هنستخدم معيار HellaSwag أو Arc-Easy:

In [ ]:
# قياس أداء النماذج عمليه (بدون lm_eval)
def simple_eval_perplexity(model, tokenizer, prompt):
    """simple evaluation by comparing loss on test prompt"""
    try:
        inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs['input_ids'])
            loss = outputs.loss
        return float(loss) if loss is not None else 0.0
    except:
        return 0.0

test_prompts = ['Hello', 'What', 'The']

print('\n=== مقارنة الأداء ===', flush=True)
results = {}
for prompt in test_prompts:
    try:
        base_loss = simple_eval_perplexity(base_model, tokenizer, prompt)
        saa_loss = simple_eval_perplexity(merged_model_saa, tokenizer, prompt)
        results[prompt] = {'base': base_loss, 'saa': saa_loss}
        if base_loss > 0:
            print(f'Prompt: {prompt}')
            print(f'  Base model loss: {base_loss:.4f}')
            print(f'  SAA model loss: {saa_loss:.4f}')
            pct_change = ((saa_loss - base_loss) / base_loss * 100) if base_loss != 0 else 0
            print(f'  Change: {pct_change:.2f}%')
    except Exception as e:
        print(f'Error: {e}')

## 📈 رابعاً: مقارنة مع الدمج الخطي (LERP)

In [ ]:
def lerp_merge(base_model, finetuned_models):
    """دمج خطي بسيط للمقارنة"""
    merged = copy.deepcopy(base_model)
    alpha = 1.0 / len(finetuned_models)
    
    base_params = dict(base_model.named_parameters())
    merged_params = dict(merged.named_parameters())
    ft_param_maps = [dict(ft.named_parameters()) for ft in finetuned_models]
    
    for idx, (name, base_param) in enumerate(base_params.items()):
        for ft_params in ft_param_maps:
            if name in ft_params and ft_params[name].shape == base_param.shape:
                merged_params[name].data = merged_params[name].data + (ft_params[name] - base_param).to(merged_params[name].dtype) * alpha
        if idx % 20 == 0:
            gc.collect()
    
    return merged

merged_model_lerp = lerp_merge(base_model, [math_model, code_model])

print('\n=== مقارنة LERP ===', flush=True)
for prompt in test_prompts:
    try:
        lerp_loss = simple_eval_perplexity(merged_model_lerp, tokenizer, prompt)
        if prompt in results:
            saa_loss = results[prompt]['saa']
            print(f'Prompt: {prompt}')
            print(f'  SAA loss: {saa_loss:.4f}')
            print(f'  LERP loss: {lerp_loss:.4f}')
            winner = 'SAA' if saa_loss < lerp_loss else 'LERP'
            print(f'  Winner: {winner}')
    except Exception as e:
        print(f'Error: {e}')

# تحرير الذاكرة
del math_model, code_model, base_model
gc.collect()
print('\nدور التقييم خلص!')

## 🎯 خامساً: تحليل النتائج المتوقع

بعد تشغيل الكود، هيكون عندنا ثلاثة أرقام:

    الدقة الأساسية (Base): خط أساس للنموذج قبل أي دمج

    دقة SAA: أداء خوارزميتنا الجديدة

    دقة LERP: أداء الدمج التقليدي

النتائج المحتملة:

    تفوق SAA: دليل قوي على أصالة وفعالية الخوارزمية

    تساوي الأداء: الخوارزمية صحيحة لكنها لا تقدم تحسينًا

    تراجع SAA: الخوارزمية بها ثغرات تحتاج لإصلاح

## 📝 ملاحظات مهمة للتطبيق

    اختيار النماذج: استبدل النماذج الوهمية بنسخ حقيقية محسنة على مهام مختلفة

    الذاكرة: لو واجهت مشاكل في VRAM، استخدم TinyLlama-1.1B أو Pythia-160M

    وقت التنفيذ: توقع أن تستغرق عملية الـ SVD الأولى بعض الوقت

جاهزين لنشوف نتائج أول اختراع حقيقي لمنهجيتنا!